# Reading input data {#sec-input-memb}

The objective of this section is to read input data in different formats and create a summary of the relationships in a consistent output format.

Input formats are convenient for data entry, but not optimal for data analysis.

Different experts will work with different types of input formats.

A consistent output format is useful for designing streamlined operations and functions. This format allows better interoperability in data operations. 

The output format can be back-transformed to the input formats if required.


:::{.aside}

We start a session in _R_ and load the libraries we need for this step:

In [1]:
library(readr)
#library(readxl)
library(dplyr)
library(tidyr)
library(stringr)
library(foreach)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




We also want to declare the relative path using the package `here`:

In [2]:
here::i_am("preparation/read-input-xref-data.ipynb")

here() starts at /Users/z3529065/proyectos/EIW/TASVEG-XREF



And import custom functions:

In [3]:
source(here::here("inc","functions.R"))

:::

## Read membership tables

We have cross-referencing tables from five interpreters. All files were prepared by MGT in `CSV` format with a common header but there are two different ways of representing the data. 

Most of the files will have a paired-column format with each pair indicating the target unit and membership value (`EFG_1`, `memb_1`, etc.).

In [4]:
file_path <- here::here("xrefdata/AL_V1_100626_TASVEG_5_0.csv")
lines <- readLines(file_path, n = 15)
cat(lines, sep = "\n")

Name,Adelina_Latinovic,,,,
Date,10_06_26,,,,
email,adelina.latinovic@nre.tas.gov.au,,,,
Level 6 classification,"From Forest to Fjaeldmark: Descriptions of Tasmania's Vegetation (Edition 2) (including revisions between August 2018 and February 2026): Kitchener, A. & Harris, S. (2013). From Forest to Fjaeldmark​: Descriptions of Tasmania's Vegetation. Edition 2. Department of Primary Industries, Parks, Water and Environment, Tasmania.",,,,
Level 6 classification url,Available at https://nre.tas.gov.au/conservation/flora-of-tasmania/from-forest-to-fjaeldmark-descriptions-of-tasmanias-vegetation,,,,
classification unit,Community (three letter code),,,,
number of units,156,,,,
Spatial layer,NCH TASVEG 5.0 The digital vegetation map of Tasmania,,,,
Spatial layer url,https://listdata.thelist.tas.gov.au/opendata/,,,,
join_field,VEGCODE,,,,
Comments,First cross-references of Tasmanian vegetation communities to GET Level 3,,,,
,,,,,
VEGCODE,EFG_1 ,memb_1,EFG_2,memb_2,sum
AAP,TF1.7,1,NA,NA,1
AHF,

In this case we need a function that can reshape the data from this paired-column format to a long format:

Two files have a matrix format with echo column indicating a different target unit, and cell values indicating the membership:

In [5]:
file_path <- here::here("xrefdata/AK_V1_050726_TASVEG_5_0.csv")
lines <- readLines(file_path, n = 15)
cat(lines, sep = "\n")

Name,Anne Kitchener,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Date,05_07_26,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
email,Anne.Kitchener@nre.tas.gov.au,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Level 6 classification,"From Forest to Fjaeldmark: Descriptions of Tasmania's Vegetation (Edition 2) (including revisions between August 2018 and February 2026): Kitchener, A. & Harris, S. (2013). From Forest to Fjaeldmark​: Descriptions of Tasmania's Vegetation. Edition 2. Department of Primary Industries, Parks, Water and Environment, Tasmania.",,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Level 6 classification url,Available at https://nre.tas.gov.au/conservation/flora-of-tasmania/from-forest-to-fjaeldmark-descriptions-of-tasmanias-vegetation,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
classification unit,Community (three letter code),,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
number of units,156,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Spatial layer,NCH TASVEG 5.0 The digital vegetation map of Tasmania,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Spatial layer

For these, we will use another function to reformat from matrix to long format.

We use the following code to run through the list of files, read metadata and data, apply the relevant functions according to the shape of the input data, and combine all in a single table.

In [6]:

memb_data <- foreach (j = dir(here::here("xrefdata")), .combine = bind_rows) %do% {
    metadata <- read_csv(here::here("xrefdata",j), 
        col_names = c("name", "value"), n_max=11, show_col_types = F)[,1:2]
    data <- read_csv(here::here("xrefdata",j), skip = 12, show_col_types = F)
    
    cat(sprintf("Read %s, interpreter %s\n%s\n", j, metadata[1,2], metadata[11,2]))

    if ("EFG_1" %in% colnames(data)) {
        pivot_wide_membership_table(
            xref = data,
            original_names = "VEGCODE",
            target_names = sprintf("EFG_%s", 1:4),
            membership_values = sprintf("memb_%s", 1:4),
            expert_name = metadata[[1,2]],
            round = if_else(grepl("First",metadata[11,2]), 1, 2))
    } else {
        pivot_membership_matrix(
            xref = data,
            original_names = "VEGCODE",
            target_names = colnames(data)[-1],
            expert_name = metadata[[1,2]],
            round = if_else(grepl("First",metadata[11,2]), 1, 2))
    }
}

Read AK_V1_050726_TASVEG_5_0.csv, interpreter Anne Kitchener
First formal documentation of cross-references with insights gained in discussions with David Keith, Mark Tozer (UNSW), Adelina Latinovic and Jayne Balmer (Tasmanian government NRE) covering each of the vegetation communities over which there was disagreement
Read AL_V1_100626_TASVEG_5_0.csv, interpreter Adelina_Latinovic
First cross-references of Tasmanian vegetation communities to GET Level 3
Read AL_V2_050726_TASVEG_5_0.csv, interpreter Adelina_Latinovic
Revised cross-references following discussions with David Keith, Mark Tozer (UNSW), Anne Kitchener and Jayne Balmer (Tasmanian government NRE) covering each of the vegetation communities over which there was disagreement
Read DAK_V1_240626_TASVEG_5_0.csv, interpreter David_Keith
First cross-referencing of Tasmanian Vegetation Communities to GET level 3
Read DAK_V2_120826_TASVEG_5_0.csv, interpreter David_Keith
Revised cross-references following discussions with Mark Tozer,

The output has five columns:

- `VEGCODE` is the original code of the units used in the TASVEG 5.0 dataset
- `EFG` is the code/name for the target Ecosystem Functional Group
- `memb` is the membership value between 0 and 1
- `expert` is the name of the interpreter
- `round` indicates whether the record comes from their first assessment (value of 1), or if the record comes from a revised assessment after group discussions (value of 2)

Lets look at a random sample from this table:

In [7]:
slice_sample(memb_data, n = 5)

VEGCODE,EFG,memb,expert,round
<chr>,<chr>,<dbl>,<chr>,<dbl>
HHW,TF1.6,0,Adelina_Latinovic,2
NBS,T3.4,0,Adelina_Latinovic,2
DRO,T2.3,0,Anne Kitchener,1
WDA,MT2.1,0,Anne Kitchener,1
SSW,T2.5,1,Adelina_Latinovic,1


We can summarise the data, excluding all `NA` values:

In [8]:
memb_data |> 
    filter(!is.na(memb), memb > 0) |>
    group_by(expert, round) |>
    summarise(`source ids`=n_distinct(VEGCODE), `target ids` = n_distinct(EFG), .groups = "drop")

expert,round,source ids,target ids
<chr>,<dbl>,<int>,<int>
Adelina_Latinovic,1,157,22
Adelina_Latinovic,2,156,35
Anne Kitchener,1,156,35
David_Keith,1,156,30
David_Keith,2,156,32
Jayne_Balmer,1,98,13
Jayne_Balmer,2,156,32
Mark_Tozer,1,156,29
Mark_Tozer,2,156,29


:::{.aside}

Note that we only have one file for Anne Kitchener, metadata seems to indicate this should be round 2 instead of round 1. Need to confirm with MGT.

:::

:::{.aside}
We save the membership table in a file that we can load in later steps of the workflow:

In [9]:
output_dir <- 
  here::here("output-XREFs")
if (!dir.exists(output_dir))
  dir.create(output_dir)

TASVEG_xref <- memb_data |> 
    filter(
      !is.na(memb), 
      memb > 0)

save(TASVEG_xref, 
file = here::here(output_dir, 
  "memb-tables.rda"))

In [10]:
#| echo: false
#| label: export anonimised data
interpreters <- c("Worrysome Wren", "Blissful Badger", "Frightened Flamingo", "Pointless Panda", "Tantalising Tarantula")
names(interpreters) <- sample(c("Adelina_Latinovic","David_Keith","Jayne_Balmer","Mark_Tozer","Anne Kitchener"))

TASVEG_xref |>
transmute(
    VEGCODE,
    EFG,
    memb,
    interp=interpreters[expert],
    round
) |>
write.csv(here::here(output_dir,"TASVEG-xref-two-rounds-anon.csv"))

:::